# pycatdap チュートリアル — AICによるカテゴリカルデータ解析

**CATDAP**（CATegorical Data Analysis Program）は、赤池弘次のAIC（赤池情報量規準）をカテゴリカルデータの分析に適用した手法です。統計数理研究所の坂元慶行・桂光一により1980年に開発されました。

`pycatdap` はR版 `catdap` パッケージのPython実装であり、以下の2つの主要機能を提供します：

| 関数 | 目的 |
|------|------|
| `catdap1()` | 全変数ペア間の関連度をΔAICで評価 |
| `catdap2()` | 最適説明変数の部分集合探索（連続変数の自動カテゴリ化を含む） |

## このノートブックの構成

1. **数理的基礎** — AIC統計量の定義と解釈
2. **データ探索** — 同梱データセットの確認
3. **CATDAP-01** — カテゴリカル変数ペアの関連度分析 + 可視化
4. **CATDAP-02** — 最適説明変数探索（プーリング含む）+ 可視化
5. **大規模データ** — HelloGoodbyeデータセットでの実行
6. **正当性検証** — 結果の整合性チェック

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pycatdap
from pycatdap.datasets import load_health_data, load_hello_goodbye
from pycatdap.plotting import aic_comparison_plot, barplot_twoway, mosaic_plot

%matplotlib inline
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 11
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"pycatdap version: {pycatdap.__version__}")

## 1. 数理的基礎

### AIC統計量（分割表モデル）

目的変数 $E$、説明変数 $F$ に対するAIC：

$$AIC(E; F) = -2 \sum_{i,j} n_{EF}(i,j) \ln \frac{n_{EF}(i,j)}{n_F(j)} + 2(C_E - 1) C_F$$

- $n_{EF}(i,j)$: クロス度数（目的変数カテゴリ $i$、説明変数カテゴリ $j$）
- $n_F(j)$: 説明変数の周辺度数
- $C_E, C_F$: それぞれのカテゴリ数

### ベースAIC（説明変数なしモデル）

$$AIC(E; \phi) = -2 \sum_i n_E(i) \ln \frac{n_E(i)}{n} + 2(C_E - 1)$$

### ΔAIC（出力値）

$$\Delta AIC = AIC(E; F) - AIC(E; \phi)$$

| ΔAIC の値 | 解釈 |
|-----------|------|
| $\Delta AIC < 0$ | 説明変数 $F$ は目的変数の説明に**有効** |
| $\Delta AIC \geq 0$ | 説明変数 $F$ は**無用**（ペナルティがゲインを上回る） |

> **注意**: ゼロ度数セルは $0 \times \ln(0) = 0$ として処理します。

## 2. データ探索

### HealthData（52例、8変数）

R版 `catdap` パッケージ同梱の医療データ。カテゴリカル変数と連続変数が混在しています。

| 変数 | 型 | 説明 |
|------|-----|------|
| opthalmo. | カテゴリカル (1, 2) | 眼科検査結果 |
| ecg | カテゴリカル (1, 2) | 心電図結果 |
| **symptoms** | カテゴリカル (A, B) | **症状分類（目的変数）** |
| age | 連続 | 年齢 |
| max.press | 連続 | 最高血圧 |
| min.press | 連続 | 最低血圧 |
| aortic.wav | 連続 | 大動脈波 |
| cholesterol | カテゴリカル (low, high) | コレステロール |

In [ ]:
df = load_health_data()
print(f"Shape: {df.shape}")
df.head(10)

In [ ]:
df.describe(include="all").round(2)

## 3. CATDAP-01 — カテゴリカル変数ペアの関連度分析

`catdap1()` は**カテゴリカル変数のみ**を対象とし、全変数ペア間のΔAICを計算します。

- `response_names=None`（デフォルト）: 全変数を順に目的変数として評価
- `response_names=["symptoms"]`: 指定した変数のみを目的変数として評価

> **注意**: 連続変数をそのまま渡すと、ユニーク値の数だけカテゴリが生成され、ペナルティ項が過大になります。連続変数には `catdap2()` のプーリング機能を使ってください。

In [ ]:
# カテゴリカル列のみを抽出して catdap1 を実行
cat_df = df[["symptoms", "opthalmo.", "ecg", "cholesterol"]]

result1 = pycatdap.catdap1(cat_df)
print(result1)
print("\n--- ΔAIC matrix ---")
result1.aic

In [ ]:
# 症状 (symptoms) に対する説明変数のランキング
print("symptoms に対する説明変数ランキング（ΔAIC昇順 = 良い順）:")
for i, var in enumerate(result1.aic_order["symptoms"], 1):
    aic_val = result1.aic.loc["symptoms", var]
    marker = "◎" if aic_val < 0 else "×"
    print(f"  {i}. {var:15s}  ΔAIC = {aic_val:+.4f}  {marker}")

### 可視化

ΔAICの棒グラフ（緑 = 有効、赤 = 無用）と、最良変数の二元分割表を可視化します。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ΔAIC 比較棒グラフ
aic_comparison_plot(result1, response="symptoms", ax=axes[0])

# 最良変数の二元分割表 → 帯グラフ
best_var = result1.aic_order["symptoms"][0]
tway = result1.tway_tables[("symptoms", best_var)]
barplot_twoway(tway, ax=axes[1])
axes[1].set_title(f"symptoms × {best_var}")

# モザイクプロット
mosaic_plot(tway, ax=axes[2])
axes[2].set_title(f"Mosaic: symptoms × {best_var}")

plt.tight_layout()
plt.show()

### ΔAICの非対称性

ΔAICは対称ではありません: $\Delta AIC(\text{symptoms} \to \text{ecg}) \neq \Delta AIC(\text{ecg} \to \text{symptoms})$

In [ ]:
aic_s_e = result1.aic.loc["symptoms", "ecg"]
aic_e_s = result1.aic.loc["ecg", "symptoms"]
print(f"ΔAIC(symptoms → ecg)   = {aic_s_e:+.4f}")
print(f"ΔAIC(ecg → symptoms)   = {aic_e_s:+.4f}")
print(f"差分 = {abs(aic_s_e - aic_e_s):.4f}  → 非対称であることを確認")

## 4. CATDAP-02 — 最適説明変数部分集合探索

`catdap2()` は `catdap1()` を拡張し、以下の機能を追加します：

1. **連続変数のプーリング** — AIC最小化により連続変数を最適なカテゴリに自動分割
2. **多変数部分集合探索** — 1変数, 2変数, ... と変数を追加しながら最適組み合わせを探索

### pool パラメータ

| 値 | 意味 | 対象 |
|----|------|------|
| `0` | 等間隔プーリング（トップダウン） | 連続変数 |
| `1` | 不等間隔プーリング（ボトムアップ、デフォルト） | 連続変数 |
| `2` | プーリングなし | カテゴリカル変数 |

HealthData の設定:
```
pool = [2, 2, 2, 0, 0, 0, 0, 2]
       opthalmo. ecg symptoms age max.press min.press aortic.wav cholesterol
       ↑cat     ↑cat ↑cat    ↑cont ↑cont   ↑cont    ↑cont      ↑cat
```

In [ ]:
result2 = pycatdap.catdap2(
    df,
    pool=[2, 2, 2, 0, 0, 0, 0, 2],
    response_name="symptoms",
    accuracy=[0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.1, 0.0],
)
print(result2)

In [ ]:
# ベースAIC（説明変数なしモデル）
print(f"Base AIC (null model): {result2.base_aic:.4f}")
print(f"\n--- 単変数 ΔAIC ランキング ---")
result2.aic

In [ ]:
# ΔAIC 比較棒グラフ
fig, ax = plt.subplots(figsize=(8, 4))
aic_comparison_plot(result2, ax=ax)
plt.tight_layout()
plt.show()

### プーリング結果

連続変数がどのように最適カテゴリ化されたかを確認します。

In [ ]:
print("--- プーリング境界 ---")
for var, boundaries in result2.intervals.items():
    n_bins = len(boundaries) + 1
    print(f"\n  {var}: {n_bins} bins")
    if boundaries:
        print(f"    boundaries: {[round(b, 2) for b in boundaries]}")
    else:
        print("    (single bin — no split)")

### 最適部分集合

変数数ごとの最良の説明変数組み合わせとΔAICを確認します。

In [ ]:
print("--- 最適部分集合 ---")
seen_nvars = set()
for s in result2.subsets:
    if s.n_vars not in seen_nvars:
        seen_nvars.add(s.n_vars)
        vars_str = ", ".join(s.variables)
        print(f"  {s.n_vars}変数: [{vars_str}]  ΔAIC = {s.aic:+.4f}  (categories={s.n_categories})")

## 5. 大規模データ — HelloGoodbye

HelloGoodbyeは13,954行、56個のバイナリ変数からなる大規模データセットです。`nvar` パラメータで探索対象変数数を制限することで計算時間を制御できます。

In [ ]:
df_hg = load_hello_goodbye()
print(f"Shape: {df_hg.shape}")
print(f"Response: Isay — values: {sorted(df_hg['Isay'].unique())}")
print(f"All binary: {all(set(df_hg[c].unique()) <= {0, 1} for c in df_hg.columns)}")
df_hg.head()

In [ ]:
%%time
# 全変数バイナリなので pool=2, nvar=5 で探索を制限
pool_hg = [2] * len(df_hg.columns)
result_hg = pycatdap.catdap2(
    df_hg, pool=pool_hg, response_name="Isay", nvar=5
)
print(result_hg)

In [ ]:
# 上位10変数のΔAICを表示
print("--- Isay に対する上位10変数 ---")
result_hg.aic.head(10)

In [ ]:
# 上位変数のΔAIC可視化（上位15変数のみ表示）
top15 = result_hg.aic.head(15)
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["tab:green" if v < 0 else "tab:red" for v in top15["aic"]]
ax.barh(top15["variable"], top15["aic"], color=colors)
ax.set_xlabel("ΔAIC")
ax.set_title("HelloGoodbye: Top 15 variables for Isay")
ax.axvline(0, color="black", linewidth=0.8)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6. 正当性検証

以下のチェックにより、pycatdap の出力が数理的に正しいことを確認します。

R版 catdap パッケージとの厳密な数値比較には `docs/r_reference/generate_reference.R` を実行して生成した CSV を使用できます。ここではR環境がなくても確認できる**構造的・数理的な整合性チェック**を行います。

In [ ]:
from pycatdap._aic import compute_aic_twoway, compute_base_aic, compute_delta_aic
from pycatdap._contingency import build_crosstab

checks = []

# --- Check 1: ΔAIC = AIC(E;F) - AIC(E;φ) の恒等式 ---
cross, marg_e, marg_f, n = build_crosstab(cat_df, "symptoms", "cholesterol")
delta = compute_delta_aic(cross, marg_e, marg_f, n)
twoway = compute_aic_twoway(cross, marg_f)
base = compute_base_aic(marg_e, n)
ok1 = np.isclose(delta, twoway - base, rtol=1e-10)
checks.append(("ΔAIC = AIC(E;F) - AIC(E;φ) の恒等式", ok1))

# --- Check 2: 完全に独立な変数 → ΔAIC ≥ 0 ---
rng = np.random.default_rng(42)
ind_df = pd.DataFrame({
    "Y": rng.choice(["a", "b"], size=10000),
    "X": rng.choice(["p", "q", "r"], size=10000),
})
r_ind = pycatdap.catdap1(ind_df, response_names=["Y"])
ok2 = r_ind.aic.loc["Y", "X"] > -1.0  # approximately non-negative
checks.append(("独立変数 → ΔAIC ≈ 0（非負）", bool(ok2)))

# --- Check 3: 完全関連 → ΔAIC << 0 ---
perf_df = pd.DataFrame({
    "Y": ["a"] * 50 + ["b"] * 50,
    "X": ["p"] * 50 + ["q"] * 50,
})
r_perf = pycatdap.catdap1(perf_df, response_names=["Y"])
ok3 = r_perf.aic.loc["Y", "X"] < -10.0
checks.append(("完全関連 → ΔAIC << 0", bool(ok3)))

# --- Check 4: ベースAIC > 0 ---
ok4 = result2.base_aic > 0
checks.append(("ベースAIC > 0（非自明なデータ）", bool(ok4)))

# --- Check 5: 全ΔAIC値が有限 ---
ok5 = bool(np.all(np.isfinite(result2.aic["aic"].to_numpy())))
checks.append(("全ΔAIC値が有限（NaN/Inf なし）", ok5))

# --- Check 6: aortic.wav が catdap2 の最良単変数 ---
ok6 = result2.aic_order[0] == "aortic.wav"
checks.append(("catdap2: aortic.wav が最良単変数", bool(ok6)))

# --- Check 7: プーリング境界が連続変数に存在 ---
cont_vars = ["age", "max.press", "min.press", "aortic.wav"]
ok7 = all(v in result2.intervals for v in cont_vars)
checks.append(("連続変数にプーリング境界が存在", ok7))

# --- Check 8: 部分集合に2変数以上が含まれる ---
ok8 = max(s.n_vars for s in result2.subsets) >= 2
checks.append(("部分集合探索が2変数以上に到達", bool(ok8)))

# --- Check 9: catdap1 ΔAIC の非対称性 ---
ok9 = result1.aic.loc["symptoms", "ecg"] != result1.aic.loc["ecg", "symptoms"]
checks.append(("catdap1 ΔAIC の非対称性", bool(ok9)))

# --- Check 10: 0*ln(0)=0 の検証 ---
from pycatdap._aic import _safe_xlogy
ok10 = float(_safe_xlogy(np.array([0.0]), np.array([0.0]))[0]) == 0.0
checks.append(("0 × ln(0) = 0 の規約", ok10))

# --- 結果表示 ---
print("=" * 55)
print("正当性検証サマリー")
print("=" * 55)
all_ok = True
for name, passed in checks:
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"  {status}  {name}")
    if not passed:
        all_ok = False
print("=" * 55)
if all_ok:
    print("全 10 項目 PASS — 結果は正当です")
else:
    print("⚠️ 一部の検証が失敗しました")

## 参考文献

- 赤池弘次 (1974). "A new look at the statistical model identification." *IEEE Transactions on Automatic Control*, 19(6), 716-723.
- 坂元慶行, 桂光一 (1980). "カテゴリカルデータの解析プログラム CATDAP." 統計数理研究所.
- R `catdap` パッケージ: https://cran.r-project.org/package=catdap

---

### R版との厳密な数値比較

R環境がある場合は `docs/r_reference/generate_reference.R` を実行して CSV を生成し、Python の出力と比較できます：

```r
Rscript docs/r_reference/generate_reference.R
```

生成される CSV:
- `health_catdap1.csv` — catdap1 の ΔAIC（カテゴリカル列）
- `health_catdap2_aic.csv` — catdap2 の単変数 ΔAIC
- `health_catdap2_subsets.csv` — catdap2 の最適部分集合